# Gold Layer — dim_product

Joins CRM `product_info` with ERP `product_info2` into a single `workspace.gold.dim_product` dimension table.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Product Key Normalisation** | Strip the 2-segment category/subcategory prefix from `product_key` (e.g., `CO-RF-FR-R92B-58` → `FR-R92B-58`) using `REGEXP_EXTRACT` so it matches the short form used in `silver.sales_details` |
| 2 | **Direct Key Join** | `Category_key` in silver `product_info` already uses underscores (e.g., `CO_RF`), matching `category_id` from ERP directly — no normalisation needed |
| 3 | **LEFT JOIN** | Joined CRM `product_info` (base, 397 rows) with ERP `product_info2` on `Category_key = category_id` to enrich products with category, subcategory, and maintenance info |
| 4 | **Deduplication** | Silver source has multiple SCD-style versions per `product_key` (different `start_date`/`end_date`/`is_active`). `ROW_NUMBER()` keeps only one row per `product_key` — preferring `is_active = true`, then latest `start_date` — reducing 397 rows → 295 unique products |
| 5 | **Unified Product Dimension** | Combined CRM product details (name, cost, dates, active flag) with ERP category metadata into a single gold dimension table with one row per product |

---

## Source Tables

| Source Table | Layer | Columns Contributed |
|---|---|---|
| `workspace.silver.product_info` | CRM | product_id, product_key, Category_key, product_name, cost, product_line, start_date, end_date, is_active |
| `workspace.silver.product_info2` | ERP (px_cat_g1v2) | category, subcategory, maintenance_required |

---

## Join Logic

```
CRM product_info (base, 397 rows)
  LEFT JOIN silver.product_info2 (px_cat_g1v2)
    ON pi.Category_key = pi2.category_id
```

| Metric | Count |
|---|---|
| Total Silver Rows | 397 |
| Distinct product_keys (after dedup) | 295 |
| Matched with product_info2 | 290 |
| Unmatched (CRM only) | 5 |

---

## Implementation

All joins are implemented using **SQL** with LEFT JOIN to preserve all CRM product records. The `Category_key` (already underscore-separated in silver) joins directly with `category_id` from ERP — no key transformation required.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_product
COMMENT 'Gold-layer product dimension table joining CRM product_info and ERP product_info2 silver tables. Deduplicated to one row per product_key.'
AS
WITH product_enriched AS (
    SELECT
        pi.product_id,
        REGEXP_EXTRACT(pi.product_key, '^[^-]+-[^-]+-(.*)$', 1) AS product_key,
        pi.Category_key,
        pi.product_name,
        pi.cost,
        pi.product_line,
        pi.start_date,
        pi.end_date,
        pi.is_active,
        pi2.category,
        pi2.subcategory,
        pi2.maintenance_required,
        ROW_NUMBER() OVER (
            PARTITION BY REGEXP_EXTRACT(pi.product_key, '^[^-]+-[^-]+-(.*)$', 1)
            ORDER BY pi.is_active DESC, pi.start_date DESC
        ) AS rn
    FROM workspace.silver.product_info pi
    LEFT JOIN workspace.silver.product_info2 pi2
        ON pi.Category_key = pi2.category_id
)
SELECT
    product_id,
    product_key,
    Category_key,
    product_name,
    cost,
    product_line,
    start_date,
    end_date,
    is_active,
    category,
    subcategory,
    maintenance_required
FROM product_enriched
WHERE rn = 1

In [0]:
%sql
SELECT * FROM workspace.gold.dim_product LIMIT 10